# 13.3 CTC and sequence transduction

CTC learns from unsegmented sequences by summing the probabilities of every frame-level alignment that collapses to the target.

CTC keeps frame softmax evidence but changes the loss: every monotonic alignment that collapses to the transcript contributes probability mass.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1301)


def sine_wave(freq, duration, fs, amplitude=1.0, phase=0.0):
    n = np.arange(int(round(duration * fs)))
    x = amplitude * np.sin(2.0 * np.pi * freq * n / fs + phase)
    return x


def chirp_wave(start_freq, end_freq, duration, fs, amplitude=1.0):
    n = np.arange(int(round(duration * fs)))
    t = n / fs
    slope = (end_freq - start_freq) / duration
    phase = 2.0 * np.pi * (start_freq * t + 0.5 * slope * t * t)
    x = amplitude * np.sin(phase)
    return x


def rms(x):
    return float(np.sqrt(np.mean(np.square(x))))


def add_noise_for_snr(x, snr_db, seed):
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(size=len(x))
    target_noise_rms = rms(x) / (10.0 ** (snr_db / 20.0))
    noise = noise / (rms(noise) + 1e-12)
    y = x + target_noise_rms * noise
    return y


def frame_signal(x, n_fft, hop):
    if len(x) < n_fft:
        pad_width = n_fft - len(x)
        x = np.pad(x, (0, pad_width))
    starts = np.arange(0, len(x) - n_fft + 1, hop)
    frames = np.stack([x[start:start + n_fft] for start in starts])
    return frames


def stft_mag(x, n_fft, hop):
    frames = frame_signal(x, n_fft, hop)
    window = np.hanning(n_fft)
    spectrum = np.fft.rfft(frames * window[None, :], axis=1)
    magnitude = np.abs(spectrum)
    return magnitude


def hz_to_mel(freq):
    return 2595.0 * np.log10(1.0 + freq / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def mel_filterbank(fs, n_fft, n_mels):
    freqs = np.linspace(0.0, fs / 2.0, n_fft // 2 + 1)
    mel_edges = np.linspace(hz_to_mel(0.0), hz_to_mel(fs / 2.0), n_mels + 2)
    hz_edges = mel_to_hz(mel_edges)
    bank = np.zeros((n_mels, len(freqs)))
    for band in range(n_mels):
        left = hz_edges[band]
        center = hz_edges[band + 1]
        right = hz_edges[band + 2]
        rising = (freqs - left) / max(center - left, 1e-12)
        falling = (right - freqs) / max(right - center, 1e-12)
        bank[band] = np.maximum(0.0, np.minimum(rising, falling))
    return bank


def dct_matrix(n):
    rows = []
    for k in range(n):
        row = []
        for b in range(n):
            value = math.cos(math.pi * k * (2 * b + 1) / (2 * n))
            row.append(value)
        rows.append(row)
    return np.array(rows)


def build_audio_ladder(fs=8000):
    d1 = sine_wave(440.0, 0.45, fs)
    d2 = 0.65 * sine_wave(440.0, 0.50, fs) + 0.35 * sine_wave(660.0, 0.50, fs)
    d3_clean = 0.60 * sine_wave(440.0, 0.55, fs) + 0.35 * sine_wave(880.0, 0.55, fs)
    d3 = add_noise_for_snr(d3_clean, 12.0, 1303)
    d4 = np.concatenate([
        sine_wave(330.0, 0.20, fs),
        chirp_wave(360.0, 900.0, 0.25, fs),
        0.80 * sine_wave(550.0, 0.20, fs),
        0.65 * sine_wave(720.0, 0.15, fs),
    ])
    d5_clean = np.concatenate([
        0.70 * sine_wave(300.0, 0.25, fs),
        chirp_wave(320.0, 980.0, 0.35, fs),
        0.55 * sine_wave(440.0, 0.25, fs) + 0.25 * sine_wave(880.0, 0.25, fs),
        0.70 * sine_wave(660.0, 0.30, fs),
        chirp_wave(900.0, 420.0, 0.25, fs),
    ])
    d5 = add_noise_for_snr(d5_clean, 5.0, 1305)
    ladder = [
        {"name": "D1 single synthetic sine", "x": d1, "fs": fs, "targets": [440.0], "complexity": 1},
        {"name": "D2 two-tone command", "x": d2, "fs": fs, "targets": [440.0, 660.0], "complexity": 2},
        {"name": "D3 noisy two-tone", "x": d3, "fs": fs, "targets": [440.0, 880.0], "complexity": 3},
        {"name": "D4 chirp multi-segment", "x": d4, "fs": fs, "targets": [330.0, 550.0, 720.0], "complexity": 4},
        {"name": "D5 longer noisier phrase", "x": d5, "fs": fs, "targets": [300.0, 440.0, 660.0], "complexity": 5},
    ]
    return ladder


def dominant_frequencies(x, fs, n_fft=512, hop=128, count=3):
    mag = stft_mag(x, n_fft, hop)
    mean_mag = np.mean(mag, axis=0)
    mean_mag[0] = 0.0
    order = np.argsort(mean_mag)[::-1]
    freqs = order[:count] * fs / n_fft
    return freqs


def nearest_error(predicted, targets):
    predicted = np.array(predicted)
    errors = []
    for target in targets:
        error = float(np.min(np.abs(predicted - target)))
        errors.append(error)
    return float(np.mean(errors))


## The concept, built once (D1): path collapse

CTC paths use symbols plus blank $\varnothing$.  The collapse map $B(\pi)$ first merges consecutive repeats, then removes blanks.

In [ ]:

BLANK = "_"
ALPHABET = [BLANK, "a", "b"]

def ctc_collapse(path, blank=BLANK):
    merged = []
    previous = None
    for symbol in path:
        if symbol != previous:
            merged.append(symbol)
        previous = symbol
    collapsed = [symbol for symbol in merged if symbol != blank]
    return tuple(collapsed)

lesson_probs = np.array([
    [0.6, 0.3, 0.1],
    [0.5, 0.4, 0.1],
    [0.2, 0.6, 0.2],
])
path = [BLANK, BLANK, "a"]
path_indices = [ALPHABET.index(symbol) for symbol in path]
path_probability = float(np.prod([lesson_probs[t, idx] for t, idx in enumerate(path_indices)]))

assert round(path_probability, 3) == 0.18
assert ctc_collapse(path) == ("a",)

print("path probability", round(path_probability, 3))
print("collapse", ctc_collapse(path))


The CTC sequence probability is

$$P(y\mid x)=\sum_{\pi:B(\pi)=y}\prod_{t=1}^{T}p_t(\pi_t), \qquad \mathcal{L}_{CTC}=-\log P(y\mid x).$$

The code implements exact enumeration for teaching and a dynamic program for the same monotonic sum.

In [ ]:

def enumerate_ctc_score(probs, target, alphabet, blank=BLANK):
    total = 0.0
    valid_paths = []
    for path in itertools.product(alphabet, repeat=probs.shape[0]):
        if ctc_collapse(path, blank) == tuple(target):
            probability = 1.0
            for t, symbol in enumerate(path):
                probability *= probs[t, alphabet.index(symbol)]
            total += probability
            valid_paths.append((path, probability))
    return total, valid_paths

def ctc_score(probs, target, alphabet, blank=BLANK):
    extended = [blank]
    for symbol in target:
        extended.append(symbol)
        extended.append(blank)
    state_count = len(extended)
    alpha = np.zeros((probs.shape[0], state_count))
    alpha[0, 0] = probs[0, alphabet.index(extended[0])]
    if state_count > 1:
        alpha[0, 1] = probs[0, alphabet.index(extended[1])]
    for t in range(1, probs.shape[0]):
        for s in range(state_count):
            total = alpha[t - 1, s]
            if s - 1 >= 0:
                total += alpha[t - 1, s - 1]
            if s - 2 >= 0 and extended[s] != blank and extended[s] != extended[s - 2]:
                total += alpha[t - 1, s - 2]
            alpha[t, s] = total * probs[t, alphabet.index(extended[s])]
    probability = alpha[-1, -1] + alpha[-1, -2]
    return float(probability), alpha, extended

enum_total, valid_paths = enumerate_ctc_score(lesson_probs, ["a"], ALPHABET)
dp_total, alpha, extended = ctc_score(lesson_probs, ["a"], ALPHABET)
loss = -np.log(dp_total)
raw_path_count = len(ALPHABET) ** lesson_probs.shape[0]

assert raw_path_count == 27
assert round(float(enum_total), 3) == 0.498
assert round(float(dp_total), 3) == 0.498
assert round(float(loss), 3) == 0.697
assert np.round(alpha[-1], 4).tolist() == [0.06, 0.396, 0.102]

print("raw paths", raw_path_count)
print("valid paths", len(valid_paths))
print("P(a|x)", round(float(dp_total), 3))
print("state masses", np.round(alpha[-1], 4).tolist())


## The dataset ladder

The same inline F7 audio ladder becomes CTC frame probabilities for labels a, ab, noisy ab, a chirped word, and a longer repeated-letter segment.

In [ ]:

def make_ctc_ladder(fs=8000):
    base = build_audio_ladder(fs)
    labels = [
        ["a"],
        ["a", "b"],
        ["a", "b"],
        ["a", "b"],
        ["a", "a", "b"],
    ]
    ladder = []
    for rung, label in zip(base, labels):
        item = dict(rung)
        item["target"] = label
        ladder.append(item)
    return ladder

ctc_ladder = make_ctc_ladder()

for rung in ctc_ladder:
    print(rung["name"])
    print("  waveform", rung["x"].shape)
    print("  target", "".join(rung["target"]))
    print("  preview", np.round(rung["x"][:6], 3).tolist())


## Run the SAME method across D1-D5

The same CTC collapse and dynamic-programming score are applied to every rung, and the metric is sequence exact-match accuracy.

In [ ]:

def ctc_probs_from_audio(x, fs, n_fft=256, hop=96):
    mag = stft_mag(x, n_fft, hop)
    freqs = np.arange(n_fft // 2 + 1) * fs / n_fft
    a_weight = np.exp(-0.5 * ((freqs - 440.0) / 70.0) ** 2)
    b_weight = np.exp(-0.5 * ((freqs - 660.0) / 80.0) ** 2)
    energy = np.mean(mag, axis=1)
    blank_score = np.percentile(energy, 60) - energy
    a_score = mag @ a_weight
    b_score = mag @ b_weight
    logits = np.stack([blank_score, a_score, b_score], axis=1)
    logits = logits / np.maximum(np.std(logits, axis=1, keepdims=True), 1e-6)
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exps / np.sum(exps, axis=1, keepdims=True)
    return probs

def best_path_decode(probs, alphabet):
    ids = np.argmax(probs, axis=1)
    path = [alphabet[i] for i in ids]
    return list(ctc_collapse(path))

ctc_rows = []

for rung in ctc_ladder:
    probs = ctc_probs_from_audio(rung["x"], rung["fs"])
    sequence, alpha, expanded = ctc_score(probs, rung["target"], ALPHABET)
    decoded = best_path_decode(probs, ALPHABET)
    exact = int(decoded == rung["target"])
    ctc_rows.append({"rung": rung, "probs": probs, "alpha": alpha, "decoded": decoded, "metric": exact, "score": sequence})

print("rung | sequence exact-match accuracy | CTC score | best-path decoded")
for row in ctc_rows:
    print(row["rung"]["name"], "|", row["metric"], "|", round(row["score"], 4), "|", "".join(row["decoded"]))


## Results visualization

The closing figure has two parts: spectrogram/alignment-heatmap panels for every rung, then sequence accuracy vs complexity.

In [ ]:

fig, axes = plt.subplots(5, 2, figsize=(10, 12))
acc = []

for idx, row in enumerate(ctc_rows):
    rung = row["rung"]
    mag = stft_mag(rung["x"], rung["fs"], 256, 96)
    axes[idx, 0].imshow(20.0 * np.log10(mag.T + 1e-6), origin="lower", aspect="auto", cmap="magma")
    axes[idx, 0].set_title(rung["name"] + " spectrogram")
    axes[idx, 1].imshow(row["probs"].T, origin="lower", aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
    axes[idx, 1].set_yticks([0, 1, 2])
    axes[idx, 1].set_yticklabels(ALPHABET)
    axes[idx, 1].set_title("CTC alignment probabilities")
    acc.append(row["metric"])

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), acc, marker="o")
plt.xlabel("complexity rung")
plt.ylabel("sequence exact-match accuracy")
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.show()


## Pitfall on D5: best-path probability underestimates the sequence

The lesson path blank, blank, a is only one alignment.  On repeated-letter D5, argmax-only decoding can miss probability mass that the CTC sum recovers.

In [ ]:

d5 = ctc_rows[-1]
d5_probs = d5["probs"]
argmax_ids = np.argmax(d5_probs, axis=1)
best_path_probability = float(np.prod(d5_probs[np.arange(len(argmax_ids)), argmax_ids]))
summed_probability, d5_alpha, d5_extended = ctc_score(d5_probs, d5["rung"]["target"], ALPHABET)
argmax_sequence = best_path_decode(d5_probs, ALPHABET)
argmax_exact = int(argmax_sequence == d5["rung"]["target"])
summed_exact = int(summed_probability > best_path_probability)

print("argmax sequence", "".join(argmax_sequence), "exact", argmax_exact)
print("best path probability", best_path_probability)
print("summed CTC probability", summed_probability)
print("summed probability is larger", summed_exact)


## Evaluate it + Practice

- Metric: sequence exact-match accuracy across D1-D5
- No-skill baseline: guess the most common/simple output and compare against the ladder metric.
- Cheap sanity check: enumerating 27 raw paths for target a should give P(a|x)=0.498 and loss 0.697
- Ablation: replace CTC summation with only the argmax path; sequence probability should be too small
- Failure signals: repeat letters collapse away, best-path confidence is tiny, or non-monotonic targets are requested

Practice prompts:

**Practice.** List the valid paths for target ab with three frames.

**Practice.** Change blank probability and observe how the CTC score changes.

**Practice.** Construct a repeated-letter target and test why blanks are necessary.